In [ ]:
import os
import time
import zipfile
from pathlib import Path
from collections import Counter

import gdown
import matplotlib.pyplot as plt
import numpy as np

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

%matplotlib inline

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Устройство для обучения:', DEVICE)

# **Получение данных**

In [ ]:
DATA_URL = 'https://storage.yandexcloud.net/aiueducation/Content/base/l8/diseases.zip'
ARCHIVE_NAME = 'diseases.zip'

gdown.download(DATA_URL, ARCHIVE_NAME, quiet=True)

with zipfile.ZipFile(ARCHIVE_NAME, 'r') as archive:
    archive.extractall()

print('Архив загружен и распакован.')

# **Чтение текстов и разбиение на обучение/проверку**

In [ ]:
DATA_DIR = Path('dis')
TRAIN_RATIO = 0.8

CLASS_LIST = []
text_train = []
text_test = []


def read_text_file(file_path):
    """Читает текстовый файл. Если UTF-8 не подходит, используется CP1251."""
    try:
        return file_path.read_text(encoding='utf-8')
    except UnicodeDecodeError:
        return file_path.read_text(encoding='cp1251')


for file_path in sorted(DATA_DIR.iterdir()):
    if file_path.suffix.lower() != '.txt':
        continue

    disease_name = file_path.stem
    CLASS_LIST.append(disease_name)

    print(f'Добавление файла "{file_path.name}" в класс "{disease_name}"')

    full_text = read_text_file(file_path)
    words = full_text.replace('\n', ' ').split()
    split_index = int(len(words) * TRAIN_RATIO)

    train_part = ' '.join(words[:split_index])
    test_part = ' '.join(words[split_index:])

    text_train.append(train_part)
    text_test.append(test_part)

CLASS_COUNT = len(CLASS_LIST)
print('Количество классов:', CLASS_COUNT)
print(CLASS_LIST)

# **Контрольный вывод текстов**

In [ ]:
for class_index, class_name in enumerate(CLASS_LIST):
    print(f'Класс: {class_name}')
    print(f'  train: {text_train[class_index][:300]}')
    print(f'  test : {text_test[class_index][:300]}')
    print()

# **Контекстный менеджер для замера времени**

In [ ]:
class WorkTimer:
    def __enter__(self):
        self.start_time = time.time()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        duration = time.time() - self.start_time
        print(f'Время обработки: {duration:.2f} с')

# **Параметры обработки текста**

In [ ]:
VOCAB_SIZE = 5000
WIN_SIZE = 50
WIN_HOP = 5

BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 0.001

TOKEN_FILTERS = '!”#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'
print('Размер словаря:', VOCAB_SIZE)
print('Размер окна:', WIN_SIZE)
print('Шаг окна:', WIN_HOP)

# **Токенизатор без TensorFlow/Keras**

In [ ]:
class SimpleTokenizer:


    def __init__(self, num_words, filters=TOKEN_FILTERS, lower=True):
        self.num_words = num_words
        self.filters = filters
        self.lower = lower
        self.word_index = {}

    def _prepare_words(self, text):
        if self.lower:
            text = text.lower()

        replace_map = str.maketrans({symbol: ' ' for symbol in self.filters})
        cleaned_text = text.translate(replace_map)
        return cleaned_text.split()

    def fit_on_texts(self, texts):
        word_counter = Counter()
        first_position = {}

        for text in texts:
            for word in self._prepare_words(text):
                if word not in first_position:
                    first_position[word] = len(first_position)
                word_counter[word] += 1

        sorted_words = sorted(
            word_counter.keys(),
            key=lambda word: (-word_counter[word], first_position[word]),
        )

        # Индекс 0 оставляем под пустые значения.
        self.word_index = {
            word: index + 1
            for index, word in enumerate(sorted_words[:self.num_words - 1])
        }

    def texts_to_sequences(self, texts):
        sequences = []

        for text in texts:
            sequence = []
            for word in self._prepare_words(text):
                word_id = self.word_index.get(word)
                if word_id is not None and word_id < self.num_words:
                    sequence.append(word_id)
            sequences.append(sequence)

        return sequences


tokenizer = SimpleTokenizer(num_words=VOCAB_SIZE)
tokenizer.fit_on_texts(text_train)

seq_train = tokenizer.texts_to_sequences(text_train)
seq_test = tokenizer.texts_to_sequences(text_test)

print('Размер построенного словаря:', len(tokenizer.word_index))

# **Функция для нарезки текстов на окна**

In [ ]:
def make_text_windows(sequences, window_size, step):
    """Нарезает последовательности слов на окна и формирует числовые метки классов."""
    x_data = []
    y_data = []

    for class_id, sequence in enumerate(sequences):
        last_start = len(sequence) - window_size + 1

        if last_start <= 0:
            continue

        for start in range(0, last_start, step):
            finish = start + window_size
            x_data.append(sequence[start:finish])
            y_data.append(class_id)

    return np.array(x_data, dtype=np.int64), np.array(y_data, dtype=np.int64)

# **Формирование обучающей и проверочной выборок**

In [ ]:
x_train, y_train = make_text_windows(seq_train, WIN_SIZE, WIN_HOP)
x_test, y_test = make_text_windows(seq_test, WIN_SIZE, WIN_HOP)

print('Форма x_train:', x_train.shape)
print('Форма y_train:', y_train.shape)
print('Форма x_test :', x_test.shape)
print('Форма y_test :', y_test.shape)

# **DataLoader для PyTorch**

In [ ]:
def make_loader(x_data, y_data, batch_size, shuffle=False):
    x_tensor = torch.tensor(x_data, dtype=torch.long)
    y_tensor = torch.tensor(y_data, dtype=torch.long)

    dataset = TensorDataset(x_tensor, y_tensor)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
    )


train_loader = make_loader(x_train, y_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = make_loader(x_test, y_test, batch_size=BATCH_SIZE, shuffle=False)

# **Модель на PyTorch**

In [ ]:
class DiseaseTextCNN(nn.Module):

    def __init__(self, vocab_size, class_count, embedding_dim=64):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0,
        )

        self.spatial_dropout = nn.Dropout1d(p=0.3)
        self.batch_norm = nn.BatchNorm1d(num_features=embedding_dim)

        self.conv_block = nn.Sequential(
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=64,
                kernel_size=5,
                padding=2,
            ),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(p=0.3),
            nn.Conv1d(
                in_channels=64,
                out_channels=32,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
        )

        self.classifier = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(64, class_count),
        )

    def forward(self, x):
        # x имеет вид [размер_пакета, длина_окна]
        x = self.embedding(x)

        x = x.permute(0, 2, 1)

        x = self.spatial_dropout(x)
        x = self.batch_norm(x)
        x = self.conv_block(x)

        x = torch.max(x, dim=2).values
        x = self.classifier(x)

        return x


model = DiseaseTextCNN(
    vocab_size=VOCAB_SIZE,
    class_count=CLASS_COUNT,
).to(DEVICE)

print(model)

# **Функции обучения и проверки**

In [ ]:
def run_one_epoch(model, data_loader, loss_function, optimizer=None):
    is_train_stage = optimizer is not None

    if is_train_stage:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for x_batch, y_batch in data_loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        if is_train_stage:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train_stage):
            logits = model(x_batch)
            loss = loss_function(logits, y_batch)

            if is_train_stage:
                loss.backward()
                optimizer.step()

        batch_size = y_batch.size(0)
        predicted_classes = logits.argmax(dim=1)

        total_loss += loss.item() * batch_size
        total_correct += (predicted_classes == y_batch).sum().item()
        total_count += batch_size

    mean_loss = total_loss / total_count
    mean_accuracy = total_correct / total_count

    return mean_loss, mean_accuracy


def train_model(model, train_loader, test_loader, epochs, learning_rate):
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    history = {
        'loss': [],
        'accuracy': [],
        'val_loss': [],
        'val_accuracy': [],
    }

    for epoch in range(1, epochs + 1):
        train_loss, train_accuracy = run_one_epoch(
            model=model,
            data_loader=train_loader,
            loss_function=loss_function,
            optimizer=optimizer,
        )

        with torch.no_grad():
            test_loss, test_accuracy = run_one_epoch(
                model=model,
                data_loader=test_loader,
                loss_function=loss_function,
                optimizer=None,
            )

        history['loss'].append(train_loss)
        history['accuracy'].append(train_accuracy)
        history['val_loss'].append(test_loss)
        history['val_accuracy'].append(test_accuracy)

        print(
            f'Эпоха {epoch:02d}/{epochs} | '
            f'loss: {train_loss:.4f} | accuracy: {train_accuracy:.4f} | '
            f'val_loss: {test_loss:.4f} | val_accuracy: {test_accuracy:.4f}'
        )

    return history

# **Обучение модели**

In [ ]:
with WorkTimer():
    history = train_model(
        model=model,
        train_loader=train_loader,
        test_loader=test_loader,
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
    )

# **Графики точности и ошибки**

In [ ]:
def show_training_graphs(history):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    axes[0].plot(history['accuracy'], label='Обучающая выборка')
    axes[0].plot(history['val_accuracy'], label='Проверочная выборка')
    axes[0].set_title('График точности')
    axes[0].set_xlabel('Эпоха')
    axes[0].set_ylabel('Точность')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(history['loss'], label='Обучающая выборка')
    axes[1].plot(history['val_loss'], label='Проверочная выборка')
    axes[1].set_title('График ошибки')
    axes[1].set_xlabel('Эпоха')
    axes[1].set_ylabel('Ошибка')
    axes[1].legend()
    axes[1].grid(True)

    plt.show()


show_training_graphs(history)

# **Проверка распознавания классов**

In [ ]:
def collect_predictions(model, data_loader):
    model.eval()

    predictions = []
    true_labels = []

    with torch.no_grad():
        for x_batch, y_batch in data_loader:
            x_batch = x_batch.to(DEVICE)
            logits = model(x_batch)
            batch_predictions = logits.argmax(dim=1).cpu().numpy()

            predictions.extend(batch_predictions.tolist())
            true_labels.extend(y_batch.numpy().tolist())

    return np.array(true_labels), np.array(predictions)


true_labels, predicted_labels = collect_predictions(model, test_loader)
recognized_classes = sorted(set(true_labels[true_labels == predicted_labels]))

print('Правильно распознанные классы:', len(recognized_classes))
for class_id in recognized_classes:
    print(f'- {CLASS_LIST[class_id]}')

# **Матрица ошибок**

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels, labels=list(range(CLASS_COUNT)))

plt.figure(figsize=(10, 10))
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_LIST,
)
display.plot(xticks_rotation=45, cmap=None)
plt.title('Матрица ошибок')
plt.show()